In [ ]:
# | default_exp report.analysis

# Report Analysis
> Analyze articles for metadata duplicates, links, heading structure, and SEO metrics.

In [ ]:
# | export
from sqlmodel import Session, select
from seootter.article import get_articles_by_website, Article
from seootter.parser.metadata import (
    parse_metadata,
    parse_notebook_metadata,
    remove_metadata,
    extract_notebook_content, extract_html_metadata,
)
from seootter.parser.checks import check_desc_length, check_content_length, check_title_length
from seootter.parser.extractors import extract_headers, extract_images, extract_links, filter_external_links, filter_internal_links, imgs_missing_alts
from seootter.parser.utils import fetch_url_as_markdown
from seootter.content.checks import check_h1_count, check_keyword_placement, content_freshness
import httpx

FETCH = "::fetch::"


In [ ]:
# | export

def _collect_metadata_field(session: Session,  # Active database session
                            website_id: int,  # Parent website ID
                            field: str  # Metadata field to extract
                            ) -> dict[str, str]:
    "Collect a specific metadata field across all articles for a website."
    values = {}
    for article in get_articles_by_website(session, website_id):
        if article.file_path == FETCH:
            continue
        try:
            content = open(article.file_path).read()
        except FileNotFoundError:
            continue
        try:
            meta = parse_notebook_metadata(content) if article.file_path.endswith(".ipynb") else parse_metadata(content)
            values[article.file_path] = meta.get(field, "")
        except Exception:
            continue
    return values


In [ ]:
# | export

def analyze_article(article: Article,
                    domain: str,
                    is_quarto: bool,
                    title_is_h1: bool = False,
                    desc_field: str = "description",
                    title_field: str = "title",
                    date_field: str = "date",
                    fetch_base_url: str | None = None,
                    prefetched: dict | None = None,
                    ) -> dict:
    if article.file_path == FETCH:
        if prefetched and article.url in prefetched:
            p = prefetched[article.url]
            metadata = p["metadata"]
            content = p["content"]
            headers = p["headers"]
        else:
            try:
                fetch_url = article.url.replace(f"https://{domain}", fetch_base_url) if fetch_base_url else article.url
                html = httpx.get(fetch_url, timeout=15).text
                metadata = extract_html_metadata(html)
                content = fetch_url_as_markdown(fetch_url)
                headers = extract_headers(content)
            except Exception:
                return {"file_path": article.file_path, "error": "fetch_failed"}
    elif article.file_path.endswith(".ipynb"):
        try:
            raw = open(article.file_path).read()
        except FileNotFoundError:
            return {"file_path": article.file_path, "error": "file_not_found"}
        metadata = parse_notebook_metadata(raw)
        content = extract_notebook_content(raw, is_quarto=is_quarto)
        headers = extract_headers(content)
    else:
        try:
            raw = open(article.file_path).read()
        except FileNotFoundError:
            return {"file_path": article.file_path, "error": "file_not_found"}
        metadata = parse_metadata(raw)
        content = remove_metadata(raw)
        headers = extract_headers(content)

    try:
        return {"file_path": article.file_path,
                "title_check": check_title_length(metadata.get(title_field, "")),
                "description_check": check_desc_length(metadata.get(desc_field, "")),
                "content_check": check_content_length(content),
                "h1_check": check_h1_count(headers, title=metadata.get("title", ""), title_is_h1=title_is_h1),
                "missing_alts": imgs_missing_alts(extract_images(content)),
                "link_analysis": analyze_links(content, domain),
                "keyword_placement": check_keyword_placement(article.focus_keyword, metadata=metadata,
                                                             content=content, headers=headers,
                                                             url=article.url),
                "metadata": metadata}
    except Exception:
        return {"file_path": article.file_path, "error": "analysis_failed"}


In [ ]:
# | export
def analyze_links(content: str,  # Page content
                  domain: str  # Site domain to classify links
                  ) -> dict:
    "Analyze internal and external links in content."
    try:
        all_urls = list(extract_links(content).keys())
        return {"total_links": len(all_urls),
                "internal_count": len(filter_internal_links(all_urls, domain)),
                "external_count": len(filter_external_links(all_urls, domain)),
                "internal_links": filter_internal_links(all_urls, domain),
                "external_links": filter_external_links(all_urls, domain)}
    except Exception:
        return {"total_links": 0, "internal_count": 0, "external_count": 0,
                "internal_links": [], "external_links": []}


In [ ]:
# | export
def check_heading_structure(headers: list[dict]  # Headers from `extract_headers`
                            ) -> dict:
    "Check heading structure for H2 presence and skipped levels."
    try:
        levels = [int(h["type"][1]) for h in headers]
        skipped = [{"from": headers[i], "to": headers[i + 1]}
                   for i in range(len(levels) - 1) if levels[i + 1] - levels[i] > 1]
        return {"has_h2": any(h["type"] == "h2" for h in headers), "skipped_levels": skipped}
    except Exception:
        return {"has_h2": False, "skipped_levels": []}


In [ ]:
# | export
from difflib import SequenceMatcher

def find_duplicate_metadata(
    session: Session,  # Active database session
    field: str,  # Metadata field to check
    website_id: int,  # Parent website ID
    similarity_threshold: float = 0.9,  # Minimum similarity to flag
) -> list:
    "Find articles with duplicate or very similar metadata field values."
    values = _collect_metadata_field(session, website_id, field)
    items = list(values.items())
    duplicates = []
    seen = set()
    for i, (path_a, val_a) in enumerate(items):
        if not val_a or i in seen:
            continue
        group = [path_a]
        for j, (path_b, val_b) in enumerate(items[i+1:], i+1):
            if not val_b:
                continue
            try:
                ratio = SequenceMatcher(None, val_a.lower(), val_b.lower()).ratio()
            except Exception:
                continue
            if ratio >= similarity_threshold:
                group.append(path_b)
                seen.add(j)
        if len(group) > 1:
            duplicates.append({"value": val_a, "paths": group})
    return duplicates
